# CS 4650 — Homework 1: Weather Map/Reduce

HW1: 
- Install Miniconda or Anaconda on your computer, as described in Topic 5.
- Create a directory for storing this project, then create an environment, as described in Topic 5.
- Install Jupyter (Topic 6) and MRJob (Topic 9).
- Download the extra files that were used in the Topic 9 lecture, Topic_9_data.zip.  Then place the 1901 and 1902 files in the project folder (not in the 'env' subfolder).
- Create a Jupyter Notebook named "HW1_your_name.ipynb", where, of course, "your_name" is in fact your name (or the name of the person you want to give the grade to).
- In one Cell of the Notebook write the code to produce the Map/Reduce file (as shown in Topic 9), and in a subsequent Cell, write the code to run the job.
- DO NOT reset and clear the cells, so that your results will still appear in the notebook.  This saves me from having to rerun your results.
- Submit the notebook file to Canvas.  DO NOT zip the file.

In [18]:
%%file wind_dir_temp.py

import re
import json

from mrjob.job import MRJob

QUALITY_RE = re.compile(r"[01459]")

class WindDirection(MRJob):

    def mapper(self, _, line):
        val = line.strip()

        (temp, tq, wind_dir, wq) = (val[87:92], val[92:93], val[60:63], val[63:64])

        # we want direction as key, we want weather data --> temperature
        # want: low, high, count
        # group all with same direction

        # "valid" directions and temps
        if (
            temp != "+9999"
            and wind_dir != "999"
            and re.match(QUALITY_RE, tq)
            and re.match(QUALITY_RE, wq) ):  
            yield wind_dir, {"temp":temp, "count":1}
            # key              value
            # if: yield key, {values}



    def reducer(self, key, values):
        #  the reducer will find the min max total. from group -> single line per direction
        count = 0
        low = None
        high = None

        for v in values:
            temp1 = v["temp"]

            if low is None:
                low = temp1
            else: 
                low = min(low, temp1)

            if high is None:
                high = temp1
            else: 
                high = max(high, temp1)

            count = count + 1
            
        yield key, {"low": low, "high": high, "count": count}

if __name__ == '__main__':
    WindDirection.run()

Overwriting wind_dir_temp.py


In [20]:
!python wind_dir_temp.py -r local "Topic_9_Data (1)/1901" "Topic_9_Data (1)/1902"

No configs found; falling back on auto-configuration
No configs specified for local runner
Creating temp directory C:\Users\anici\AppData\Local\Temp\wind_dir_temp.anici.20261002.010954.561140
Running step 1 of 1...
Error while reading from C:\Users\anici\AppData\Local\Temp\wind_dir_temp.anici.20261002.010954.561140\step\000\mapper\00000\input:

Step 1 of 1 failed: [WinError 2] The system cannot find the file specified
